In [1]:
import plotly
import numpy as np
import plotly.graph_objs as go

In [2]:
# consider a fully actuated sheet (i.e., all the joints are controllable)
import pinocchio as pin
SDF_PATH="/home/ds3a/dev/wadiyan_carpet/models/mesh/mass_mesh_open_tree.sdf"
# model, constraints = pin.buildModelFromSdf("mass_mesh_open_tree.sdf", root_link_name="mass_x0_y0")
model, constraints = pin.buildModelFromSdf(SDF_PATH, root_link_name="mass_x0_y0")
# model = pin.buildModelFromSdf("spring_dampers_open_tree.sdf", root_link_name="mass_x0_y0")
# model = pin.buildModelFromSdf("mass_mesh_open_tree.sdf")
data = model.createData()

In [3]:
len(model.frames)

50

In [4]:
# Store all the frame IDs for a grid of masses in a dictionary
# stores the IDs for masses named mass_x{i}_y{j}

frame_ids = dict()

num_elements_x = 5
num_elements_y = 5


for i in range(int(-num_elements_x/2), int(num_elements_x/2)+1):
    for j in range(int(-num_elements_y/2), int(num_elements_y/2)+1):
        frame_ids[(i, j)] = model.getFrameId(f"mass_x{i}_y{j}")

In [ ]:
q = pin.neutral(model)
# q = pin.randomConfiguration(model)
pin.forwardKinematics(model, data, q)
link_positions = []
for i in range(int(-num_elements_x/2), int(num_elements_x/2)+1):
    for j in range(int(-num_elements_y/2), int(num_elements_y/2)+1):
        pin.updateFramePlacement(model, data, frame_ids[(i, j)])
        # print(f"Frame mass_x{i}_y{j} placement:\n{data.oMf[frame_ids[(i, j)]].translation}\n")
        link_positions.append(data.oMf[frame_ids[(i, j)]].translation)

In [2]:
side_length = 0.4
u_min, u_max = -side_length/2, side_length/2
v_min, v_max = -side_length/2, side_length/2

# Resolution of the grid (higher = smoother)
num_u = 50
num_v = 50

u = np.linspace(u_min, u_max, num_u)
v = np.linspace(v_min, v_max, num_v)

goals_u = np.linspace(u_min, u_max, num_elements_x)
goals_v = np.linspace(v_min, v_max, num_elements_y)

U, V = np.meshgrid(u, v)
goals_U, goals_V = np.meshgrid(goals_u, goals_v)

A = 0.05
angle = np.pi
phase = np.pi + np.pi/5
phase = 0.1
frequency = np.pi/0.4


# 2. Define your gamma(u, v)
def gamma_sur(u, v, A=0.5, base_pos=np.array([0, 0, 0])):
    x = u
    y = v
    s = u * np.cos(angle) + v * np.sin(angle)

    # z = A * np.sin(0.7*v + np.pi/6) * np.cos(u) # bump
    z = A * np.cos(frequency*s + phase) # bump

    # TODO modify x, y, and z such that (0, 0) lies at the base_pos
    offset = base_pos - np.array([0, 0, A * np.cos(frequency*(0) + phase)])
    #                                                          s = 0 at (0,0)
    x += offset[0]
    y += offset[1]
    z += offset[2]

    return x, y, z


goal_positions = dict()



X, Y, Z = gamma_sur(U, V, A=A)
goals_X, goals_Y, goals_Z = gamma_sur(goals_U, goals_V, A=A)

# 3. Make the Plotly surface figure
fig = go.Figure(data=[
    go.Surface(x=X, y=Y, z=Z),
    # go.Scatter3d(x=goals_X.flatten(), y=goals_Y.flatten(), z=goals_Z.flatten(), 
            # mode='markers', marker=dict(size=23, color='red')),
    go.Scatter3d(x=[pos[0] for pos in link_positions],y=[pos[1] for pos in link_positions],z=[pos[2] for pos in link_positions],
            mode='markers', marker=dict(size=12, color='blue')),
])

fig.update_layout(
    title="γ(u, v): bump surface",
    scene=dict(
        aspectmode='data',
        xaxis_title="u",
        yaxis_title="v",
        zaxis_title="height"
    )
)

# 4. Show the interactive plotmass_mesh_open_tree
# fig.show()

plotly.io.write_html(fig, file="bump_surface.html", auto_open=True)


NameError: name 'np' is not defined

In [8]:
# constraints need to be formulated from WE and EW
# we will create a list of frames that need to be linked via constraints
constraint_frames = []
for i in range(-int(num_elements_x/2), int(num_elements_x/2)+1):
    for j in range(-int(num_elements_y/2), int(num_elements_y/2)):
        if i != 0:
            constraint_frames.append([(i, j), (i, j+1)])

len(constraint_frames)

# number of constraints = (n-1)^2

16

In [ ]:
elem_dist = side_length / (num_elements_x - 1)

def get_cJacobian(model, data, q, frame_id1, frame_id2, dist):
    pin.computeJointJacobians(model, data, q)
    pin.updateFramePlacements(model, data)

    pin.forwardKinematics(model, data, q)

    J1 = pin.getFrameJacobian(model, data, frame_id1, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
    J2 = pin.getFrameJacobian(model, data, frame_id2, pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)

    p1 = data.oMf[frame_id1].translation
    p2 = data.oMf[frame_id2].translation

    constraint_fn_der_wrt_FK = 2*(p1 - p2).reshape(1, 3)
    # Constraint: c(q) = ||p1(q) - p2(q)||^2
    # dc/dq = 2*(p1 - p2)^T * (J2 - J1)
    # (sign is arbitrary for constraints)

    # relative Jacobian
    J_rel = J2 - J1
    J_rel = J_rel[0:3, :]  # only position constraints
    # print(J_rel.shape, constraint_fn_der_wrt_FK.shape)
    J_const = constraint_fn_der_wrt_FK.dot(J_rel)
    return J_const

def get_cJacobian_for_constraint(model, data, q, constraint_pair, dist):
    frame_id1 = frame_ids[constraint_pair[0]]
    frame_id2 = frame_ids[constraint_pair[1]]
    return get_cJacobian(model, data, q, frame_id1, frame_id2, dist)

def get_cJacobian_matrix(model, data, q, constraint_frames, dist):
    J_c_list = []
    for constraint_pair in constraint_frames:
        J_c = get_cJacobian_for_constraint(model, data, q, constraint_pair, dist)
        J_c_list.append(J_c)
        # print(np.linalg.det(J_c.dot(J_c.T)))
        # must be non zero for a non-singular constraint
    J_c_matrix = np.vstack(J_c_list)
    return J_c_matrix

In [10]:
Jc = get_cJacobian_matrix(model, data, q, constraint_frames, elem_dist)
Jc.shape # should be ((n-1)^2, model.nv)

(16, 72)

In [11]:
goal_positions = dict()
# goal_positions_other = dict()
for i in range(int(-num_elements_x/2), int(num_elements_x/2)+1):
    for j in range(int(-num_elements_y/2), int(num_elements_y/2)+1):
        x, y, z = gamma_sur(i*elem_dist, j*elem_dist, A=A, base_pos=np.array([0, 0, 0]))
        goal_positions[(i, j)] = np.array([x, y, z])
        # goal_positions_other[(i, j)] = np.array([goals_X[j + int(num_elements_y/2), i + int(num_elements_x/2)],
        #                                    goals_Y[j + int(num_elements_y/2), i + int(num_elements_x/2)],
        #                                    goals_Z[j + int(num_elements_y/2), i + int(num_elements_x/2)]])

In [12]:
# Now, to come up with task jacobians

def get_task_jacobians(model, data, q, goal_positions, frame_ids):
    pin.computeJointJacobians(model, data, q)
    pin.updateFramePlacements(model, data)

    pin.forwardKinematics(model, data, q)

    J_t_list = []
    for frame_id in frame_ids:
        # print(frame_ids[frame_id])
        # TODO get the position of the frame from the pinocchio model using q
        # Then get the goal for that frame using the index and goal_positions
        # Finally, compute the task Jacobian for that frame using the position error
        frame_pos = data.oMf[frame_ids[frame_id]].translation
        goal_pos = goal_positions[frame_id]
        err = goal_pos - frame_pos
        # print(err)


        J = pin.getFrameJacobian(model, data, frame_ids[frame_id], pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
        J = -J[0:3, :]  # only position
        # print(np.linalg.det(J.dot(J.T)))
        J_t_list.append(J)
    J_t_matrix = np.vstack(J_t_list)
    return J_t_matrix

def get_task_jacobians_w_errors(model, data, q, goal_positions, frame_ids):
    pin.computeJointJacobians(model, data, q)
    pin.updateFramePlacements(model, data)

    pin.forwardKinematics(model, data, q)

    J_t_list = []
    errors = []
    for frame_id in frame_ids:
        # print(frame_ids[frame_id])
        # TODO get the position of the frame from the pinocchio model using q
        # Then get the goal for that frame using the index and goal_positions
        # Finally, compute the task Jacobian for that frame using the position error
        frame_pos = data.oMf[frame_ids[frame_id]].translation
        goal_pos = goal_positions[frame_id]
        err = goal_pos - frame_pos
        # print(err)


        J = pin.getFrameJacobian(model, data, frame_ids[frame_id], pin.ReferenceFrame.LOCAL_WORLD_ALIGNED)
        J = -J[0:3, :]  # only position
        # print(np.linalg.det(J.dot(J.T)))
        J_t_list.append(J)
        errors.append(err)
        # print(err.shape)
    error_vec = np.hstack(errors)
    # print(error_vec.shape)
    J_t_matrix = np.vstack(J_t_list)
    return J_t_matrix, error_vec


In [13]:
Jt, errors = get_task_jacobians_w_errors(model, data, q, goal_positions, frame_ids)

Jt.shape # should be (n^2 * 3, model.nv)

(75, 72)

In [14]:
# now to make a kkt matrix

def make_kkt_matrix(Jc, Jt, reg=1e-6):
    if Jc.shape[1] != Jt.shape[1]:
        raise ValueError("Jc and Jt must have the same number of columns (variables)")
    num_vars = Jc.shape[1]
    num_constraints = Jc.shape[0]
    num_tasks = Jt.shape[0]

    G = Jt.T.dot(Jt) + reg * np.eye(num_vars)
    H = Jc

    KKT_left = np.vstack([G, H])
    KKT_right = np.vstack([H.T, np.zeros((num_constraints, num_constraints))])

    KKT_matrix = np.hstack([KKT_left, KKT_right])
    return KKT_matrix

In [15]:
KKT = make_kkt_matrix(get_cJacobian_matrix(model, data, q, constraint_frames, elem_dist), get_task_jacobians(model, data, q, goal_positions, frame_ids))

KKT.shape

(88, 88)

In [16]:
# make the IK solver
def solve_ik_step(model, data, q, constraint_frames, goal_positions, frame_ids, reg=1e-6):
    Jc = get_cJacobian_matrix(model, data, q, constraint_frames, elem_dist)
    Jt, errors = get_task_jacobians_w_errors(model, data, q, goal_positions, frame_ids)

    KKT = make_kkt_matrix(Jc, Jt, reg=reg)

    num_vars = Jc.shape[1]
    num_constraints = Jc.shape[0]
    num_tasks = Jt.shape[0]

    # right hand side
    b_top = -Jt.T.dot(errors)
    b_bottom = np.zeros((num_constraints, ))
    b = np.hstack([b_top, b_bottom])

    # solve for delta_q and lambda
    sol = np.linalg.solve(KKT, b)
    delta_q = sol[0:num_vars]
    # lambda_vals = sol[num_vars:]

    return delta_q

In [17]:
for i in range(5):
    v = solve_ik_step(model, data, q, constraint_frames, goal_positions, frame_ids)
    q = pin.integrate(model, q, v)



In [ ]:
def run_constrained_ik(
    model,
    data,
    gamma_fn,
    side_length,
    num_elements_x,
    num_elements_y,
    A=0.05,
    base_pos=np.array([0, 0, 0]),
    num_iters=5,
    reg=1e-6,
):
    """Run the full constrained IK pipeline and return the converged configuration."""

    # 1) Build frame map for the mass grid
    frame_ids = {}
    for i in range(int(-num_elements_x / 2), int(num_elements_x / 2) + 1):
        for j in range(int(-num_elements_y / 2), int(num_elements_y / 2) + 1):
            frame_ids[(i, j)] = model.getFrameId(f"mass_x{i}_y{j}")

    # 2) Build pairwise distance constraints (same structure as above)
    constraint_frames = []
    for i in range(-int(num_elements_x / 2), int(num_elements_x / 2) + 1):
        for j in range(-int(num_elements_y / 2), int(num_elements_y / 2)):
            if i != 0:
                constraint_frames.append([(i, j), (i, j + 1)])

    # 3) Create goal positions from gamma
    elem_dist = side_length / (num_elements_x - 1)
    goal_positions = {}
    for i in range(int(-num_elements_x / 2), int(num_elements_x / 2) + 1):
        for j in range(int(-num_elements_y / 2), int(num_elements_y / 2) + 1):
            x, y, z = gamma_fn(i * elem_dist, j * elem_dist, A=A, base_pos=base_pos)
            goal_positions[(i, j)] = np.array([x, y, z])

    # 4) Initialize at neutral and run constrained IK iterations
    q = pin.neutral(model)
    for _ in range(num_iters):
        Jc = get_cJacobian_matrix(model, data, q, constraint_frames, elem_dist)
        Jt, errors = get_task_jacobians_w_errors(model, data, q, goal_positions, frame_ids)

        KKT = make_kkt_matrix(Jc, Jt, reg=reg)
        num_vars = Jc.shape[1]
        num_constraints = Jc.shape[0]

        b_top = -Jt.T.dot(errors)
        b_bottom = np.zeros((num_constraints,))
        b = np.hstack([b_top, b_bottom])

        sol = np.linalg.solve(KKT, b)
        delta_q = sol[0:num_vars]
        q = pin.integrate(model, q, delta_q)

    return q, frame_ids, constraint_frames, goal_positions


q, frame_ids, constraint_frames, goal_positions = run_constrained_ik(
    model,
    data,
    gamma_sur,
    side_length=side_length,
    num_elements_x=num_elements_x,
    num_elements_y=num_elements_y,
    A=A,
    base_pos=np.array([0, 0, 0]),
    num_iters=5,
)



 ### Dynamics equation for equilibrium
$$
M(q)\dot{v} + C(q, v)v + g(q) = \sum_{i \in \sigma}{J_{\omega,i}(\sum_{j \in \mathcal{N}_i}{S(q_i \ominus q_j)})} + \sum_{i \in \sigma}{\mu(i)J_{\omega, i}{\begin{bmatrix}
0\\ 0 \\ u_i
\end{bmatrix}}}
$$


In [17]:
import casadi as ca

# we have q which is the joint angles which must be maintained by our model, which we got by using the ik solver
# we want to find forces u such that the system remains in equilibrium at q, i.e., v = 0 and \dot{v} = 0

q_ik = q.copy()

# update placements at the IK configuration
pin.forwardKinematics(model, data, q_ik)
pin.updateFramePlacements(model, data)

# neighbor list (max 8 neighbors on the grid)
# TODO this needs to be checked
def get_neighbors(index):
    i, j = index
    neighbors = []
    for dx in (-1, 0, 1):
        for dy in (-1, 0, 1):
            if dx == 0 and dy == 0:
                continue
            neighbor = (i + dx, j + dy)
            if neighbor in frame_ids:
                neighbors.append(neighbor)
    return neighbors

# stiffness matrix in tangent space (diagonal)
k_rot = 0.5
S = np.diag([k_rot, k_rot, k_rot])

# gravity term at the equilibrium configuration
# with v=0 and \dot{v}=0, dynamics reduce to g(q) = RHS
# C(q, v) v = 0 for v=0

# generalized gravity
# try:
#     g = pin.computeGeneralizedGravity(model, data, q_ik)
# except AttributeError:
#     g = pin.rnea(model, data, q_ik, np.zeros(model.nv), np.zeros(model.nv))

g = pin.computeGeneralizedGravity(model, data, q_ik)
frame_keys = list(frame_ids.keys())
num_frames = len(frame_keys)

A = np.zeros((model.nv, num_frames))
tau_stiff = np.zeros(model.nv)

for idx, key in enumerate(frame_keys):
    frame_id = frame_ids[key]
    R_i = data.oMf[frame_id].rotation

    # sum neighbor orientation errors in tangent space
    delta_sum = np.zeros(3)
    for neighbor in get_neighbors(key):
        neighbor_id = frame_ids[neighbor]
        R_j = data.oMf[neighbor_id].rotation
        # quaternion log difference (SO(3) log map)
        delta = pin.log3(R_j.T @ R_i)
        delta_sum += S @ delta

    J = pin.getFrameJacobian(model, data, frame_id, pin.ReferenceFrame.WORLD)
    Jw = J[3:6, :]
    Ju = J[0:3, :]

    # stiffness contribution (maps to generalized forces)
    tau_stiff += Jw.T @ delta_sum

    # thrust contribution for this frame: Jw^T * [0, 0, u_i]
    A[:, idx] = (Ju.T @ np.array([0.0, 0.0, 1.0])).reshape(-1)

# solve A u = g - tau_stiff using a least-squares CasADi solve
b = g - tau_stiff

u = ca.MX.sym("u", num_frames)
A_ca = ca.DM(A)
b_ca = ca.DM(b)
residual = A_ca @ u - b_ca

nlp = {"x": u, "f": 0.5 * ca.dot(residual, residual)}
solver = ca.nlpsol("solver", "ipopt", nlp, {"print_time": False, "ipopt.print_level": 1})
solver = ca.nlpsol("solver", "ipopt", nlp)
sol = solver(x0=ca.DM.zeros(num_frames))

u_opt = np.array(sol["x"]).squeeze()
print("Solved thrusts (u) per frame:")
# print(u_opt)
u_dict = dict()
for idx, key in enumerate(frame_keys):
    u_dict[key] = u_opt[idx]
    print(f"Frame {key}: Thrust u = {u_opt[idx]:.4f} N")



******************************************************************************
This program contains Ipopt, a library for large-scale nonlinear optimization.
 Ipopt is released as open source code under the Eclipse Public License (EPL).
         For more information visit https://github.com/coin-or/Ipopt
******************************************************************************

This is Ipopt version 3.14.11, running with linear solver MUMPS 5.4.1.

Number of nonzeros in equality constraint Jacobian...:        0
Number of nonzeros in inequality constraint Jacobian.:        0
Number of nonzeros in Lagrangian Hessian.............:      325

Total number of variables............................:       25
                     variables with only lower bounds:        0
                variables with lower and upper bounds:        0
                     variables with only upper bounds:        0
Total number of equality constraints.................:        0
Total number of inequality c

In [18]:
tau_stiff

array([ 0.13485095, -0.38123414,  0.07744227,  0.03948187, -0.11652362,
       -0.1697901 ,  0.13485095, -0.38123414,  0.07744227,  0.03948187,
       -0.11652362, -0.1697901 , -0.64430896, -0.65272972, -0.13065245,
       -0.11566156, -0.34003803,  0.10249447, -0.11137173,  0.31111507,
       -0.09134269, -0.19198243, -0.20890667,  0.09865952, -0.06959423,
       -0.68868267, -0.11913159, -0.75555848, -0.37623531,  0.19746071,
        0.09540867,  0.47126309, -0.24251995,  0.12138149, -0.77421486,
        0.38100644,  0.04989755, -0.28427211, -0.10698994,  0.00845949,
        0.75168551,  0.33276485, -0.64430896, -0.65272972, -0.13065245,
       -0.19198243, -0.20890667,  0.09865952, -0.06959423, -0.68868267,
       -0.11913159, -0.11566156, -0.34003803,  0.10249447, -0.11137173,
        0.31111507, -0.09134269, -0.75555848, -0.37623531,  0.1974607 ,
        0.04989755, -0.28427211, -0.10698994,  0.00845949,  0.75168551,
        0.33276485,  0.09540867,  0.47126309, -0.24251995,  0.12

In [19]:
u_opt

array([ 2.239418  , -1.70435972,  0.78664011,  3.46871815, -2.08669332,
       -2.33565406,  2.02505939,  6.5619874 , -1.11277956,  2.13400348,
        2.11064237,  7.79096099,  0.        ,  7.79096099,  2.11064237,
        2.13400348, -1.11277956,  6.5619874 ,  2.0250594 , -2.33565406,
       -2.08669332,  3.46871814,  0.78664011, -1.70435972,  2.239418  ])

In [25]:
u_dict = dict()
for idx, key in enumerate(frame_keys):
    u_dict[key] = u_opt[idx]
    print(f"Frame {key}: Thrust u = {u_opt[idx]:.4f} N")

Frame (-2, -2): Thrust u = 0.4527 N
Frame (-2, -1): Thrust u = -0.1982 N
Frame (-2, 0): Thrust u = -1.0653 N
Frame (-2, 1): Thrust u = -0.1982 N
Frame (-2, 2): Thrust u = 0.4527 N
Frame (-1, -2): Thrust u = 1.2115 N
Frame (-1, -1): Thrust u = 3.3437 N
Frame (-1, 0): Thrust u = 2.5233 N
Frame (-1, 1): Thrust u = 3.3437 N
Frame (-1, 2): Thrust u = 1.2115 N
Frame (0, -2): Thrust u = 2.7966 N
Frame (0, -1): Thrust u = 23.0424 N
Frame (0, 0): Thrust u = 0.0000 N
Frame (0, 1): Thrust u = 23.0424 N
Frame (0, 2): Thrust u = 2.7966 N
Frame (1, -2): Thrust u = 1.2115 N
Frame (1, -1): Thrust u = 3.3437 N
Frame (1, 0): Thrust u = 2.5233 N
Frame (1, 1): Thrust u = 3.3437 N
Frame (1, 2): Thrust u = 1.2115 N
Frame (2, -2): Thrust u = 0.4527 N
Frame (2, -1): Thrust u = -0.1982 N
Frame (2, 0): Thrust u = -1.0653 N
Frame (2, 1): Thrust u = -0.1982 N
Frame (2, 2): Thrust u = 0.4527 N


In [26]:
# c++ code for u_dict
# print("\nC++ code for u_dict:")
print("std::map<std::pair<int, int>, double> u_dict = {")
for idx, key in enumerate(frame_keys):
    print(f"    {{std::make_pair({key[0]}, {key[1]}), {u_opt[idx]:.6f}}},")
print("};")


std::map<std::pair<int, int>, double> u_dict = {
    {std::make_pair(-2, -2), 0.452709},
    {std::make_pair(-2, -1), -0.198225},
    {std::make_pair(-2, 0), -1.065296},
    {std::make_pair(-2, 1), -0.198225},
    {std::make_pair(-2, 2), 0.452709},
    {std::make_pair(-1, -2), 1.211462},
    {std::make_pair(-1, -1), 3.343669},
    {std::make_pair(-1, 0), 2.523286},
    {std::make_pair(-1, 1), 3.343669},
    {std::make_pair(-1, 2), 1.211462},
    {std::make_pair(0, -2), 2.796608},
    {std::make_pair(0, -1), 23.042372},
    {std::make_pair(0, 0), 0.000000},
    {std::make_pair(0, 1), 23.042372},
    {std::make_pair(0, 2), 2.796608},
    {std::make_pair(1, -2), 1.211462},
    {std::make_pair(1, -1), 3.343669},
    {std::make_pair(1, 0), 2.523286},
    {std::make_pair(1, 1), 3.343669},
    {std::make_pair(1, 2), 1.211462},
    {std::make_pair(2, -2), 0.452709},
    {std::make_pair(2, -1), -0.198225},
    {std::make_pair(2, 0), -1.065296},
    {std::make_pair(2, 1), -0.198225},
    {std::m

In [20]:
np.linalg.matrix_rank(A)

24